In [2]:
from prediLib import JIRARequest,WILCORequest,get_apu_parameters_values
import pandas as pd
import os
from pathlib import Path
from datetime import datetime,timezone
import json
import src.ML_functions as ML_functions
import src.config as config
import warnings
warnings.filterwarnings('ignore')

2026-03-18 14:12:03.561 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-03-18 14:12:03.567 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


<h2>API CLIENT INITIALIZATION & DATA FETCHING<h2>
This section will initialize wilco and jira session for data querying using PrediLib library

In [3]:
jira_base_url=os.getenv("JIRA_REVIMA_BASE_URL")
jira_user=os.getenv("JIRA_USER")
jira_token=os.getenv("JIRA_API_TOKEN")
wilco_api_key=os.getenv("API_KEY")
jira_client=JIRARequest(base_url=jira_base_url,user_name=jira_user,user_token=jira_token)
wilco_client=WILCORequest(api_key=wilco_api_key)

[ERROR] - Not able to connect to JIRA api 


In [3]:
raw_data_path=Path("data_base")/"data_for_parameters_analysis.pkl"
raw_data_checkpoint=Path("data_base")/"raw_data_checkpoint.json"
apuonwingsession_data_path=Path("data_base")/"on_wing_session.pkl"

raw_data_path.parent.mkdir(parents=True,exist_ok=True)

AT_EVENT_parameters=config.AT_EVENT_parameters

MES_parameters=config.MES_parameters

All_parameters=AT_EVENT_parameters+MES_parameters

In [4]:
if os.path.exists(raw_data_path) and os.path.exists(raw_data_checkpoint):
    with open(raw_data_checkpoint,'r') as f:
        last_timestamp=json.load(f)
    existing_data_base=pd.read_pickle(raw_data_path)
    print(f"Number of rows of existing Database : {len(existing_data_base)}")
    try:
        data_base=get_apu_parameters_values(jira_client=jira_client,wilco_client=wilco_client,fwot_type="B787",parameters=All_parameters,current_data=True,from_date=last_timestamp['last_timestamp'])
        raw_data_counts=len(data_base)
        data_base=data_base.sort_values(by=['Reg','Date'],ascending=[False,True]).reset_index(drop=True)
        data_base[All_parameters]=data_base[All_parameters].map(ML_functions.converter_to_float)
        data_base[MES_parameters]=data_base.loc[:,MES_parameters].bfill()
        data_base.dropna(subset=AT_EVENT_parameters,inplace=True)
        data_base=ML_functions.compute_features(df_samples=data_base,SMA_10_dict=config.SMA_10_cols,SMA_30_dict=config.SMA_30_cols,MED_30_dict=config.MED_30_cols,CUM_SUM_dict=config.CUM_SUM_cols)
        print("[DONE] - Fetching Data From WILCO and Data cleaning")
        print("="*60)
        print(f"Number of Rows Cleaned data : {len(data_base)}")
        print(f"Percentage of data lost during Cleaning Process : {100-(100*round(len(data_base)/raw_data_counts,2))} %")
        print("="*60)
    except Exception as e:
        print(f"[ERROR] during data extraction from JIRA and Wilco")
        print(f"{e}")
    new_data_base=pd.concat([existing_data_base,data_base],ignore_index=True)
    new_data_base.to_pickle(raw_data_path)
    print(f"Raw Database successfuly saved in  : {raw_data_path}")
    print("="*60)
    maxt_time=datetime.strftime(new_data_base.Date.max(),format="%Y-%m-%dT%H:%M:%S")
    with open(raw_data_checkpoint,'w') as f:
        json.dump({"last_timestamp":maxt_time},f,indent=4)
    print(f"Checkpoint successfuly saved in : {raw_data_checkpoint}")
else:
    try:
        raw_path=Path(raw_data_path)
        raw_path.parent.mkdir(parents=True,exist_ok=True)
        data_base=get_apu_parameters_values(jira_client=jira_client,wilco_client=wilco_client,fwot_type="B787",parameters=All_parameters,current_data=True,from_date="2020-01-01T00:00:00")
        raw_data_counts=len(data_base)
        data_base=data_base.sort_values(by=['Reg','Date'],ascending=[False,True]).reset_index(drop=True)
        data_base[All_parameters]=data_base[All_parameters].map(ML_functions.converter_to_float)
        data_base[MES_parameters]=data_base.loc[:,MES_parameters].bfill()
        data_base.dropna(subset=AT_EVENT_parameters,inplace=True)
        data_base=ML_functions.compute_features(df_samples=data_base,SMA_10_dict=config.SMA_10_cols,SMA_30_dict=config.SMA_30_cols,MED_30_dict=config.MED_30_cols,CUM_SUM_dict=config.CUM_SUM_cols)
        print("[DONE] - Fetching Data From WILCO and Data cleaning")
        print("="*60)
        print(f"Number of Rows Cleaned data : {len(data_base)}")
        print(f"Percentage of data lost during Cleaning Process : {100-(100*round(len(data_base)/raw_data_counts,2))} %")
        print("="*60)
    except Exception as e:
        print(f"[ERROR] during data extraction from JIRA and Wilco")
        print(f"{e}")
    data_base.to_pickle(raw_data_path)
    print(f"Raw Database successfuly saved in  : {raw_data_path}")
    print("="*60)
    maxt_time=datetime.strftime(data_base.Date.max(),format="%Y-%m-%dT%H:%M:%S")
    with open(raw_data_checkpoint,'w') as f:
        json.dump({"last_timestamp":maxt_time},f)
    print(f"Checkpoint successfuly saved in : {raw_data_checkpoint}")

JIRA API REQUEST
Found : 100 Issues - Page 0
Found : 100 Issues - Page 1
Found : 100 Issues - Page 2
Found : 47 Issues - Page 3
JIRA API REQUEST SUCCESSFUL
FIELD MAPPING
FIELD MAPPING SUCCESSFUL


Fetching Data from Wilco: 100%|██████████████████████████████| 347/347 [06:16<00:00,  1.08s/]



Unique fwot extracted from Jira : 128

Number of rows extracted from Wilco: 2273855


Combining Jira data with Wilco samples: 100%|██████████████████████████████| 347/347 [00:32<00:00, 10.75fwot/s]


[DONE] - Fetching Data From WILCO and Data cleaning
Number of Rows Cleaned data : 400609
Percentage of data lost during Cleaning Process : 82.0 %
Raw Database successfuly saved in  : data_base\data_for_parameters_analysis.pkl
Checkpoint successfuly saved in : data_base\raw_data_checkpoint.json
